In [2]:
import numpy as np


# ============================================================
# USER SETTINGS
# ============================================================

input_file = "MMP.gro"
output_file = "MMP_corrected.gro"


# Desired z-coordinate of the MMP phosphorus, nm
MMP_TARGET_Z = 5.250

# ============================================================
# GEOMETRIC PARAMETERS
#
# GROMACS .gro coordinates are in nm.
# Internally we use nm.
# ============================================================

P_O = 1.60 / 10.0       # 1.60 Å
O_C = 1.43 / 10.0       # 1.43 Å
O_H = 0.96 / 10.0       # 0.96 Å
C_H = 1.09 / 10.0       # 1.09 Å


# ============================================================
# VECTOR UTILITIES
# ============================================================

def unit(v):
    return v / np.linalg.norm(v)


def angle(a, b, c):
    """
    Return angle A-B-C in degrees.
    """

    v1 = a - b
    v2 = c - b

    v1 = unit(v1)
    v2 = unit(v2)

    return np.degrees(
        np.arccos(
            np.clip(np.dot(v1, v2), -1.0, 1.0)
        )
    )


def perpendicular_vector(v):
    """
    Return an arbitrary unit vector perpendicular to v.
    """

    v = unit(v)

    # Choose a vector that isn't parallel to v
    if abs(v[0]) < 0.9:
        tmp = np.array([1.0, 0.0, 0.0])
    else:
        tmp = np.array([0.0, 1.0, 0.0])

    p = tmp - np.dot(tmp, v) * v

    return unit(p)


# ============================================================
# READ EXISTING MMP.GRO
# ============================================================

with open(input_file) as f:
    lines = f.readlines()


title = lines[0].rstrip()
natoms = int(lines[1].strip())

atom_lines = lines[2:2 + natoms]
box_line = lines[2 + natoms].rstrip()


print("Input file:", input_file)
print("Number of atoms in input:", natoms)
print("Box:", box_line)


# ============================================================
# FIND EXISTING P POSITION
#
# We keep the existing P coordinate as the anchor for the
# reconstructed molecule.
# ============================================================

P = None

for line in atom_lines:

    atom_name = line[10:15].strip()

    if atom_name == "P":

        P = np.array([
            float(line[20:28]),
            float(line[28:36]),
            float(line[36:44])
        ])

        break


if P is None:
    raise ValueError(
        "Could not find atom named P in MMP.gro"
    )


print()
print()
print("Original P position:")
print(f"P = {P}")

# ------------------------------------------------------------
# Translate the entire MMP so that P is at the desired z.
#
# IMPORTANT:
# We only change the absolute position of the molecule.
# All internal bond lengths and angles remain unchanged.
# ------------------------------------------------------------

z_shift = MMP_TARGET_Z - P[2]

P[2] += z_shift

print()
print(f"Applied z translation: {z_shift:.5f} nm")
print("New P position:")
print(f"P = {P}")

# ============================================================
# PHOSPHATE GEOMETRY
#
# Four P-O bonds arranged tetrahedrally.
#
# We use the four tetrahedral directions:
#
#       O
#      /
# O - P - O
#      \
#       O
#
# Actual orientation in space is arbitrary; P remains at the
# original coordinate.
# ============================================================

tetra = np.array([
    [ 1.0,  1.0,  1.0],
    [ 1.0, -1.0, -1.0],
    [-1.0,  1.0, -1.0],
    [-1.0, -1.0,  1.0]
])

tetra = np.array([
    unit(v) for v in tetra
])


# Assign the four phosphate oxygens
#
# OME = methoxy oxygen
# OOH = phosphate hydroxyl oxygen
# O1/O2 = anionic oxygens

OME = P + P_O * tetra[0]
OOH = P + P_O * tetra[1]
O1  = P + P_O * tetra[2]
O2  = P + P_O * tetra[3]


# ============================================================
# METHOXY GROUP
#
# Construct P-O-C with approximately 120 degrees.
#
# At O:
#
#          C
#         /
#        O
#       /
#      P
#
# P-O-C = 120 degrees
# ============================================================

d_P_OME = unit(OME - P)

# Vector perpendicular to P -> OME
perp = perpendicular_vector(d_P_OME)

angle_POC = np.deg2rad(120.0)

# O -> P is -d_P_OME
d_O_C = (
    np.cos(angle_POC) * (-d_P_OME)
    + np.sin(angle_POC) * perp
)

d_O_C = unit(d_O_C)

CM = OME + O_C * d_O_C


# ============================================================
# METHYL HYDROGENS
#
# Construct a tetrahedral CH3 group around carbon.
#
# The C-O bond occupies one tetrahedral direction.
# Three C-H bonds occupy the other three directions.
# ============================================================

d_C_O = unit(OME - CM)


# Need two mutually perpendicular vectors perpendicular
# to C -> O.

u = perpendicular_vector(d_C_O)
v = unit(np.cross(d_C_O, u))


# Tetrahedral angle
theta_CH = np.deg2rad(109.4712206)

# We construct H directions such that the angle between
# C->O and C->H is tetrahedral.
#
# cos(109.47°) = -1/3

cos_theta = np.cos(theta_CH)
sin_theta = np.sin(theta_CH)


H_methyl = []

for phi in [
    0.0,
    2.0 * np.pi / 3.0,
    4.0 * np.pi / 3.0
]:

    direction = (
        cos_theta * d_C_O
        + sin_theta
        * (
            np.cos(phi) * u
            + np.sin(phi) * v
        )
    )

    direction = unit(direction)

    H = CM + C_H * direction

    H_methyl.append(H)


HM1, HM2, HM3 = H_methyl


# ============================================================
# PHOSPHATE HYDROXYL
#
# Construct P-O-H with approximately 109.5 degrees.
#
# At O:
#
#      P
#       \
#        O
#         \
#          H
#
# P-O-H = 109.5 degrees
# ============================================================

d_P_OOH = unit(OOH - P)

perp_OH = perpendicular_vector(d_P_OOH)

angle_POH = np.deg2rad(109.5)

# O -> P is -d_P_OOH
d_O_H = (
    np.cos(angle_POH) * (-d_P_OOH)
    + np.sin(angle_POH) * perp_OH
)

d_O_H = unit(d_O_H)

HO = OOH + O_H * d_O_H


# ============================================================
# ASSEMBLE THE 10 ATOMS
#
# IMPORTANT:
#
# This is the atom order expected by the correction script:
#
# 1   P
# 2   OME
# 3   OOH
# 4   O1
# 5   O2
# 6   CM
# 7   HM1
# 8   HM2
# 9   HM3
# 10  HO
# ============================================================

atoms = [
    ("P",   P),
    ("OME", OME),
    ("OOH", OOH),
    ("O1",  O1),
    ("O2",  O2),
    ("CM",  CM),
    ("HM1", HM1),
    ("HM2", HM2),
    ("HM3", HM3),
    ("HO",  HO),
]


# ============================================================
# GEOMETRY DIAGNOSTICS
# ============================================================

print()
print("=" * 60)
print("CORRECTED MMP GEOMETRY")
print("=" * 60)

print()
print("Coordinates (Å):")

for i, (name, xyz) in enumerate(atoms, start=1):

    xyz_A = xyz * 10.0

    print(
        f"{i:2d} "
        f"{name:4s} "
        f"{xyz_A[0]:10.5f} "
        f"{xyz_A[1]:10.5f} "
        f"{xyz_A[2]:10.5f}"
    )


print()
print("Bond lengths:")
print(
    f"P-O(OME) = {np.linalg.norm(P-OME)*10:.3f} Å"
)
print(
    f"P-O(OOH) = {np.linalg.norm(P-OOH)*10:.3f} Å"
)
print(
    f"P-O(O1)  = {np.linalg.norm(P-O1)*10:.3f} Å"
)
print(
    f"P-O(O2)  = {np.linalg.norm(P-O2)*10:.3f} Å"
)
print(
    f"O-C      = {np.linalg.norm(OME-CM)*10:.3f} Å"
)
print(
    f"O-H      = {np.linalg.norm(OOH-HO)*10:.3f} Å"
)

print()
print("Phosphate angles:")

print(
    f"O-OME-P = {angle(OME, P, OOH):.3f}°"
)
print(
    f"OME-P-O1 = {angle(OME, P, O1):.3f}°"
)
print(
    f"OME-P-O2 = {angle(OME, P, O2):.3f}°"
)
print(
    f"OOH-P-O1 = {angle(OOH, P, O1):.3f}°"
)
print(
    f"OOH-P-O2 = {angle(OOH, P, O2):.3f}°"
)
print(
    f"O1-P-O2 = {angle(O1, P, O2):.3f}°"
)

print()
print("Functional-group angles:")

print(
    f"P-O-C = {angle(P, OME, CM):.3f}°"
)

print(
    f"P-O-H = {angle(P, OOH, HO):.3f}°"
)

print()
print("Methyl-group angles:")

print(
    f"O-C-HM1 = {angle(OME, CM, HM1):.3f}°"
)

print(
    f"O-C-HM2 = {angle(OME, CM, HM2):.3f}°"
)

print(
    f"O-C-HM3 = {angle(OME, CM, HM3):.3f}°"
)

print(
    f"HM1-C-HM2 = {angle(HM1, CM, HM2):.3f}°"
)

print(
    f"HM1-C-HM3 = {angle(HM1, CM, HM3):.3f}°"
)

print(
    f"HM2-C-HM3 = {angle(HM2, CM, HM3):.3f}°"
)


# ============================================================
# WRITE CORRECTED GRO FILE
# ============================================================

with open(output_file, "w") as f:

    f.write("Corrected MMP - 10 atom geometry\n")
    f.write(f"{len(atoms):5d}\n")

    for i, (name, xyz) in enumerate(atoms, start=1):

        f.write(
            f"{1:5d}"
            f"{'MMP':<5s}"
            f"{name:>5s}"
            f"{i:5d}"
            f"{xyz[0]:8.3f}"
            f"{xyz[1]:8.3f}"
            f"{xyz[2]:8.3f}\n"
        )

    f.write(box_line + "\n")


print()
print("=" * 60)
print(f"WROTE: {output_file}")
print("=" * 60)

Input file: MMP.gro
Number of atoms in input: 10
Box:    4.49070   4.66690  10.00000


Original P position:
P = [2.245 2.333 5.   ]

Applied z translation: 0.25000 nm
New P position:
P = [2.245 2.333 5.25 ]

CORRECTED MMP GEOMETRY

Coordinates (Å):
 1 P      22.45000   23.33000   52.50000
 2 OME    23.37376   24.25376   53.42376
 3 OOH    23.37376   22.40624   51.57624
 4 O1     21.52624   24.25376   51.57624
 5 O2     21.52624   22.40624   53.42376
 6 CM     24.79773   24.16098   53.33098
 7 HM1    25.22606   25.16291   53.30308
 8 HM2    25.06840   23.62466   52.42148
 9 HM3    25.18413   23.62466   54.19768
10 HO     24.29765   22.59066   51.76066

Bond lengths:
P-O(OME) = 1.600 Å
P-O(OOH) = 1.600 Å
P-O(O1)  = 1.600 Å
P-O(O2)  = 1.600 Å
O-C      = 1.430 Å
O-H      = 0.960 Å

Phosphate angles:
O-OME-P = 109.471°
OME-P-O1 = 109.471°
OME-P-O2 = 109.471°
OOH-P-O1 = 109.471°
OOH-P-O2 = 109.471°
O1-P-O2 = 109.471°

Functional-group angles:
P-O-C = 120.000°
P-O-H = 109.500°

Methyl-group a